In [5]:
import psycopg2 
db_url = "postgresql://postgres:0@localhost:1024/mental_health_db_TEST"
conn = psycopg2.connect(db_url)
curr = conn.cursor()
curr.execute('SELECT current_database()  , current_user , inet_server_port();')
print(curr.fetchone())
conn.close()
curr.close()
print('Connection OK')

('mental_health_db_TEST', 'postgres', 1024)
Connection OK


In [ ]:
import os
import sys
import django

# Allow sync ORM calls in Jupyter's async runtime
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

# Make Django project importable
sys.path.insert(0, r"c:\Users\Ryan\Desktop\startup\Persian Mental Health Website\backend")

# Point Django to your PostgreSQL DB
os.environ["DATABASE_URL"] = "postgresql://postgres:0@localhost:1024/mental_health_db_TEST"
os.environ["DJANGO_SETTINGS_MODULE"] = "core.settings"

django.setup()

# Ensure required Django tables (including auth_user) exist in this DB
from django.core.management import call_command
call_command("migrate", interactive=False, verbosity=0)

# Insert/read user using Django ORM
from django.contrib.auth.models import User

user, created = User.objects.get_or_create(
    username="jupyter_test_user",
    defaults={"email": "jupyter_test_user@example.com"},
)
if created:
    user.set_password("testpassword123")
    user.save()

print("created:", created)
print("user id:", user.id)
print("username:", user.username)
print("latest users:", list(User.objects.order_by("-id").values("id", "username", "email")[:5]))

created: True
user id: 1
username: jupyter_test_user
latest users: [{'id': 1, 'username': 'jupyter_test_user', 'email': 'jupyter_test_user@example.com'}]


In [15]:
import psycopg2

db_url = "postgresql://postgres:0@localhost:1024/mental_health_db_TEST"

conn = psycopg2.connect(db_url)
cur = conn.cursor()

# 1) Create schema
cur.execute("""
CREATE TABLE IF NOT EXISTS chat_sessions (
    id SERIAL PRIMARY KEY,
    user_name VARCHAR(100) NOT NULL,
    initial_mood VARCHAR(50),
    started_at TIMESTAMP DEFAULT NOW(),
    total_tokens INTEGER DEFAULT 0
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS chat_messages (
    id SERIAL PRIMARY KEY,
    session_id INTEGER NOT NULL REFERENCES chat_sessions(id) ON DELETE CASCADE,
    seq INTEGER NOT NULL,
    role VARCHAR(20) NOT NULL CHECK (role IN ('user','assistant','system')),
    content TEXT NOT NULL,
    token_count INTEGER DEFAULT 0,
    created_at TIMESTAMP DEFAULT NOW()
);
""")

# 2) Insert one session
cur.execute("""
INSERT INTO chat_sessions (user_name, initial_mood, total_tokens)
VALUES (%s, %s, %s)
RETURNING id;
""", ("sample_user", "sad", 73))
session_id = cur.fetchone()[0]

# 3) Insert messages for that session
cur.execute("""
INSERT INTO chat_messages (session_id, seq, role, content, token_count)
VALUES
(%s, %s, %s, %s, %s),
(%s, %s, %s, %s, %s),
(%s, %s, %s, %s, %s);
""", (
    session_id, 1, "user", "سلام، امروز حالم خوب نیست.", 18,
    session_id, 2, "assistant", "متوجهم. دوست داری از چی شروع کنیم؟", 19,
    session_id, 3, "user", "از فشار کار و بی‌خوابی.", 12
))

conn.commit()

# 4) Read back what was saved
cur.execute("SELECT id, user_name, initial_mood, total_tokens FROM chat_sessions WHERE id = %s;", (session_id,))
print("SESSION:", cur.fetchone())

cur.execute("SELECT seq, role, content, token_count FROM chat_messages WHERE session_id = %s ORDER BY seq;", (session_id,))
print("MESSAGES:")
for row in cur.fetchall():
    print(row)

cur.close()
conn.close()
print("Saved successfully.")

SESSION: (1, 'sample_user', 'sad', 73)
MESSAGES:
(1, 'user', 'سلام، امروز حالم خوب نیست.', 18)
(2, 'assistant', 'متوجهم. دوست داری از چی شروع کنیم؟', 19)
(3, 'user', 'از فشار کار و بی\u200cخوابی.', 12)
Saved successfully.
